<a href="https://colab.research.google.com/github/ghada-dahdoh/Applied-natural-language-processing/blob/main/Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss

# النصوص التي نريد البحث فيها
texts = [
    "تعذر إتمام عملية الدفع داخل التطبيق رغم نجاح خصم المبلغ من البطاقة",
    "لا تصلني رسالة التحقق عند محاولة تسجيل الدخول إلى الحساب",
    "أصبح التطبيق بطيئًا جدًا ويتوقف عند فتح صفحة الخدمات",
    "أرغب في استرجاع المبلغ الذي تم خصمه مقابل عملية شراء ملغاة",
    "نسيت كلمة المرور ولا أستطيع الوصول إلى حسابي",
    "تظهر رسالة خطأ عند محاولة تحديث البيانات الشخصية",
    "تم إيقاف الحساب مؤقتًا بعد عدة محاولات تسجيل دخول فاشلة",
    "لا تظهر لي الطلبات السابقة داخل صفحة سجل العمليات"
]

# نموذج يحول النص إلى Embedding
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# تحويل النصوص إلى Embeddings
embeddings = model.encode(texts)

# بناء فهرس FAISS
index = faiss.IndexFlatL2(embeddings.shape[1])

# إضافة الـ Embeddings إلى الفهرس
index.add(embeddings)

print("عدد النصوص في FAISS:", index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.1 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

عدد النصوص في FAISS: 8


In [2]:
query = "تم سحب فلوسي لكن العملية ما اكتملت"

query_embedding = model.encode([query])

distances, indices = index.search(query_embedding, 3)

print("Query:", query)
print("\nResults:")

for rank, i in enumerate(indices[0], start=1):
    print(f"{rank}. {texts[i]}")
    print("Distance:", round(float(distances[0][rank - 1]), 4))
    print()

Query: تم سحب فلوسي لكن العملية ما اكتملت

Results:
1. تعذر إتمام عملية الدفع داخل التطبيق رغم نجاح خصم المبلغ من البطاقة
Distance: 16.0221

2. أرغب في استرجاع المبلغ الذي تم خصمه مقابل عملية شراء ملغاة
Distance: 16.1447

3. لا تظهر لي الطلبات السابقة داخل صفحة سجل العمليات
Distance: 16.3598



In [3]:
results = [
    "أصبح التطبيق بطيئًا جدًا ويتوقف عند فتح صفحة الخدمات",
    "تعذر إتمام عملية الدفع داخل التطبيق رغم نجاح خصم المبلغ من البطاقة",
    "أرغب في استرجاع المبلغ الذي تم خصمه مقابل عملية شراء ملغاة",
    "لا تصلني رسالة التحقق عند محاولة تسجيل الدخول إلى الحساب",
    "تظهر رسالة خطأ عند محاولة تحديث البيانات الشخصية",
    "نسيت كلمة المرور ولا أستطيع الوصول إلى حسابي",
    "تم إيقاف الحساب مؤقتًا بعد عدة محاولات تسجيل دخول فاشلة",
    "لا تظهر لي الطلبات السابقة داخل صفحة سجل العمليات"
]

correct_answer = "تعذر إتمام عملية الدفع داخل التطبيق رغم نجاح خصم المبلغ من البطاقة"

rank = results.index(correct_answer) + 1

recall_at_10 = 1 if rank <= 10 else 0
mrr_at_10 = 1 / rank

print("Correct Answer Rank:", rank)
print("Recall@10:", recall_at_10)
print("MRR@10:", round(mrr_at_10, 4))

Correct Answer Rank: 2
Recall@10: 1
MRR@10: 0.5


In [4]:
from sentence_transformers import CrossEncoder

query = "تم سحب فلوسي لكن العملية ما اكتملت"

results = [
    "أرغب في استرجاع المبلغ الذي تم خصمه مقابل عملية شراء ملغاة",
    "لا تصلني رسالة التحقق عند محاولة تسجيل الدخول إلى الحساب",
    "أصبح التطبيق بطيئًا جدًا ويتوقف عند فتح صفحة الخدمات",
    "تعذر إتمام عملية الدفع داخل التطبيق رغم نجاح خصم المبلغ من البطاقة",
    "لا تظهر لي الطلبات السابقة داخل صفحة سجل العمليات"
]

reranker = CrossEncoder(
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
)

pairs = [[query, text] for text in results]

scores = reranker.predict(pairs)

# ترتيب النتائج من الأعلى إلى الأقل
ranked_results = sorted(
    zip(results, scores),
    key=lambda x: float(x[1]),
    reverse=True
)

print("Re-ranked Results:\n")

for rank, (text, score) in enumerate(ranked_results, start=1):
    print(f"{rank}. Score: {float(score):.4f}")
    print("   ", text)
    print()

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Re-ranked Results:

1. Score: -6.1914
    أصبح التطبيق بطيئًا جدًا ويتوقف عند فتح صفحة الخدمات

2. Score: -6.7480
    تعذر إتمام عملية الدفع داخل التطبيق رغم نجاح خصم المبلغ من البطاقة

3. Score: -7.3527
    لا تظهر لي الطلبات السابقة داخل صفحة سجل العمليات

4. Score: -8.0898
    أرغب في استرجاع المبلغ الذي تم خصمه مقابل عملية شراء ملغاة

5. Score: -8.4917
    لا تصلني رسالة التحقق عند محاولة تسجيل الدخول إلى الحساب



In [5]:
# البيانات الموجودة عندنا
texts = [
    "تعذر إتمام عملية الدفع داخل التطبيق رغم نجاح خصم المبلغ من البطاقة",
    "لا تصلني رسالة التحقق عند محاولة تسجيل الدخول إلى الحساب",
    "أصبح التطبيق بطيئًا جدًا ويتوقف عند فتح صفحة الخدمات",
    "أرغب في استرجاع المبلغ الذي تم خصمه مقابل عملية شراء ملغاة",
    "نسيت كلمة المرور ولا أستطيع الوصول إلى حسابي",
    "تظهر رسالة خطأ عند محاولة تحديث البيانات الشخصية",
    "تم إيقاف الحساب مؤقتًا بعد عدة محاولات تسجيل دخول فاشلة",
    "لا تظهر لي الطلبات السابقة داخل صفحة سجل العمليات"
]

queries = [
    "خصموا المبلغ من حسابي لكن الدفع فشل",
    "I cannot receive the verification code",
    "كيف أضيف بطاقة جديدة إلى المحفظة؟"
]

for query in queries:

    q_emb = model.encode([query])

    distances, indices = index.search(q_emb, 1)

    best_index = indices[0][0]
    best_distance = distances[0][0]

    print("Query:", query)
    print("Best Match:", texts[best_index])
    print("Distance:", round(float(best_distance), 4))
    print("-" * 60)

Query: خصموا المبلغ من حسابي لكن الدفع فشل
Best Match: تعذر إتمام عملية الدفع داخل التطبيق رغم نجاح خصم المبلغ من البطاقة
Distance: 9.1671
------------------------------------------------------------
Query: I cannot receive the verification code
Best Match: لا تصلني رسالة التحقق عند محاولة تسجيل الدخول إلى الحساب
Distance: 16.4141
------------------------------------------------------------
Query: كيف أضيف بطاقة جديدة إلى المحفظة؟
Best Match: تعذر إتمام عملية الدفع داخل التطبيق رغم نجاح خصم المبلغ من البطاقة
Distance: 20.4937
------------------------------------------------------------
